Setup

Sets the Gurobi licence path (only if that file exists). Run from a folder containing data/ and results/.

In [1]:
import os, sys
if os.path.exists("/Users/micahgreen/gurobi.lic"):
    os.environ["GRB_LICENSE_FILE"] = "/Users/micahgreen/gurobi.lic"

Step 1: Read the data

Reads the workbook, puts every time on one clock (absolute hour = 24 × day + hour), and computes each shipment’s loading start, loaded departure, destination service start and release from its ready time, respecting opening hours.

In [2]:
"""
MODULE 1 - DATA AND TIME (owner: Micah)

Reads the workbook, puts every time on one clock (absolute hours = 24 x day + hour),
and works out each shipment's timeline from its ready time:
loading start, loaded departure, destination service start, release.

Rules handled here: R3 (loading start), R4 (loaded trip, release, a late ready time
pushes the release later), R5 (finish before the horizon), and the opening-hours
logic used by R12.
"""
import math
import pandas as pd

HOOK = 0.25   # hours to hook a trailer   (parameters sheet: hook_hours)
DROP = 0.25   # hours to drop a trailer   (parameters sheet: drop_hours)


# ---------------------------------------------------------------------------
# 1. Read the workbook
# ---------------------------------------------------------------------------
def load_instance(path):
    """Read the 7 input sheets and return one dictionary with everything."""
    wb = pd.read_excel(path, sheet_name=None)
    par = wb["parameters"].set_index("parameter")["value"]
    inst = {
        "facilities": wb["facilities"].set_index("facility_id"),
        "depots": wb["depots"].set_index("depot_id"),
        "tractors": wb["tractors"].set_index("tractor_id"),
        "shipments": wb["shipments"].set_index("shipment_id"),
        "trailers": wb["trailer_state"].set_index("trailer_id"),
        "H": 24 * int(par["horizon_end_day"]),          # horizon end, absolute hours
        "demand_days": int(par["demand_days"]),
        "late_allowed": int(par.get("late_allowed", 0)) == 1,
        "excess_allowed": int(par.get("excess_allowed", 0)) == 1,
    }
    assert float(par["hook_hours"]) == HOOK and float(par["drop_hours"]) == DROP
    # lanes: (origin, destination) -> (km, hours). Direction matters.
    inst["lanes"] = {
        (r.origin_id, r.destination_id): (float(r.distance_km), float(r.drive_hours))
        for r in wb["lanes"].itertuples()
    }
    inst["F"] = list(inst["facilities"].index)
    inst["is_main"] = inst["excess_allowed"]           # main file: penalties allowed
    return inst


# ---------------------------------------------------------------------------
# 2. Time helpers
# ---------------------------------------------------------------------------
def abs_time(day, hour):
    """Absolute time = 24 x day + hour. Day 1 at 10:00 -> 34.0"""
    return 24 * day + hour


def lane(inst, a, b):
    """(km, hours) from a to b. Same place = 0 km, 0 hours."""
    if a == b:
        return 0.0, 0.0
    return inst["lanes"][(a, b)]


def opening_hours(inst, site):
    """Daily opening interval (open, close) of a facility or a depot."""
    row = inst["facilities"].loc[site] if site in inst["facilities"].index else inst["depots"].loc[site]
    return float(row["open_hour"]), float(row["close_hour"])


def next_open(inst, site, t, duration):
    """Earliest s >= t when `site` is open long enough for a task of `duration` hours
    (finishing exactly at closing is allowed; 0-24 means always open)."""
    o, c = opening_hours(inst, site)
    if o == 0 and c == 24:
        return t
    day = math.floor(t / 24)
    for d in range(day, day + 5):
        start = max(t, 24 * d + o)
        if start + duration <= 24 * d + c + 1e-9:
            return start
    return None


# ---------------------------------------------------------------------------
# 3. A shipment's timeline from a given ready time
# ---------------------------------------------------------------------------
def shipment_times(inst, j, ready):
    """Everything that follows once the empty for shipment j is ready at `ready`.
    Loading starts at the first open moment at/after max(deadline, ready) that fits
    loading + hook (R3). A late ready time moves every later step, including the
    release at the destination (R4)."""
    s = inst["shipments"].loc[j]
    deadline = abs_time(s["day"], s["deadline_hour"])
    load_start = next_open(inst, s["origin_id"], max(deadline, ready), s["loading_hours"] + HOOK)
    loaded_depart = load_start + s["loading_hours"] + HOOK
    arrive_dest = loaded_depart + lane(inst, s["origin_id"], s["destination_id"])[1]
    service_start = next_open(inst, s["destination_id"], arrive_dest, DROP + s["unloading_hours"])
    release = service_start + DROP + s["unloading_hours"]
    return {"ready": ready, "late_hours": max(0.0, ready - deadline),
            "load_start": load_start, "loaded_depart": loaded_depart,
            "arrive_dest": arrive_dest, "service_start": service_start, "release": release}


def shipment_table(inst):
    """One row per shipment with its on-time timeline (ready = deadline)."""
    rows = []
    for j, s in inst["shipments"].iterrows():
        deadline = abs_time(s["day"], s["deadline_hour"])
        t = shipment_times(inst, j, deadline)
        if t["release"] > inst["H"]:
            raise ValueError(f"{j} cannot finish before the horizon (R5)")
        rows.append({"shipment_id": j, "origin": s["origin_id"], "destination": s["destination_id"],
                     "deadline": deadline, **t,
                     "late_rate": float(s["late_penalty_cad_per_hour"]),
                     "loaded_km": lane(inst, s["origin_id"], s["destination_id"])[0],
                     "loaded_cost": float(s["loaded_service_cost_cad"])})
    return pd.DataFrame(rows).set_index("shipment_id")




Step 2: Build the candidate tours

Builds every legal tractor day with one or two empty moves, checking shift, depot and facility hours, driving and duty limits, and hook/drop times, then prices each at its tractor’s rates. Main-file limits: hourly departures; pickups from the 6 nearest facilities to each drop; 3 cheapest tractors kept per identical tour. Two-move tours depart every 2 h. Their second drop is at one of the 2 nearest yards to the first drop (by lane distance in either direction), and their second pickup is the first drop yard itself or one of the 6 nearest sources of the second drop. A zero gap means optimal over these tours.

In [3]:
"""
MODULE 2 - TOUR BUILDER (owner: Henry)

Builds the candidate set R: every legal tractor day, with exact times and cost.
  1-move tour: depot -> f -> hook -> tow to g -> drop -> depot
  2-move tour:         depot -> f1 -> hook -> tow to g1 -> drop -> (bobtail to f2, or stay
                       at g1 when f2 = g1: a "chain") -> hook -> tow to g2 -> drop -> depot
                       (brief: the tractor may bobtail "from its previous drop" before it hooks)
Each tour becomes one yes/no variable z_r in the model, and its cost c_r is that
variable's coefficient in the objective.

Rules checked here, before Gurobi sees a tour: R10 (starts and ends at the home
depot), R11 shift + depot hours, R12 facility hours (waiting outside counts as duty),
R13 driving and duty limits, R14 one trailer at a time, hook/drop 0.25 h, lanes by
ordered pair, R15 depots tractor-only, R16 cost at this tractor's own rates.
"""
import pandas as pd


# How candidates are limited (the brief asks us to state this).
SETTINGS = {
    # simplified: every departure every 15 min, every pair, every 2-move tour
    "simplified": {"step": 0.25, "k_nearest_sources": None, "tractors_per_option": None,
                   "chains": True, "chain_next": None, "chain_step": 0.25},
    # main: hourly departures, 6 nearest sources per destination, 3 cheapest tractors per
    # identical tour; 2-move tours: the second drop is at one of the 2 nearest yards to the
    # first drop (nearest by lane distance in either direction, since lanes are not symmetric), the second pickup is one of the 6 nearest sources of that yard or the first
    # drop yard itself (chain); departures every 2 hours
    "main":       {"step": 1.00, "k_nearest_sources": 6, "tractors_per_option": 3,
                   "chains": True, "chain_next": 2, "chain_step": 2.0},
}


def facility_pairs(inst, k_nearest):
    """(source, destination) pairs to try. None = all ordered pairs;
    k = for each destination, its k nearest sources by lane distance."""
    F = inst["F"]
    if k_nearest is None:
        return [(f, g) for f in F for g in F if f != g]
    pairs = []
    for g in F:
        srcs = sorted((f for f in F if f != g), key=lambda f: (lane(inst, f, g)[0], f))
        pairs += [(f, g) for f in srcs[:k_nearest]]
    return pairs


def follow_tour(inst, row, day, depart, legs):
    """Follow one tractor day. legs = [(f, g)] or [(f1, g1), (f2, g2)] (f2 = g1 is a chain).
    Returns (moves, return_time, drive, bobtail_km, towing_km) or None if a rule breaks."""
    depot = row["home_depot_id"]
    _, d_close = opening_hours(inst, depot)
    latest_return = 24 * day + min(row["shift_end_hour"], d_close)       # R11
    pos, t, drive, bob, tow, moves = depot, depart, 0.0, 0.0, 0.0, []
    for f, g in legs:
        km, hr = lane(inst, pos, f)                 # bobtail to f (0 if already there)
        bob += km; drive += hr; t += hr
        hook = next_open(inst, f, t, HOOK)          # wait outside if closed     R12
        if hook is None:
            return None
        km, hr = lane(inst, f, g)                   # tow f -> g                 R14
        tow += km; drive += hr
        drop = next_open(inst, g, hook + HOOK + hr, DROP)                   # R12
        if drop is None:
            return None
        moves.append((f, g, hook, drop, drop + DROP))
        pos, t = g, drop + DROP
    km, hr = lane(inst, pos, depot)                 # bobtail home               R10, R15
    bob += km; drive += hr
    ret = t + hr
    if ret > latest_return + 1e-9:                  # R11
        return None
    if drive > row["max_drive_hours"] + 1e-9:       # R13
        return None
    if ret - depart > row["max_duty_hours"] + 1e-9: # R13 (waiting counts as duty)
        return None
    return moves, ret, drive, bob, tow


def price(row, depart, ret, bob, tow, n_moves):
    """R16: cost at this tractor's own rates."""
    c = {"cost_bobtail": bob * row["bobtail_cad_per_km"],
         "cost_towing": tow * row["towing_cad_per_km"],
         "cost_duty": (ret - depart) * row["duty_cad_per_hour"],
         "cost_fees": n_moves * row["move_fee_cad"]}
    c["cost"] = sum(c.values())
    return c


def build_tours(inst, settings):
    """Return (tours, moves): one row per candidate tour, one row per move inside it."""
    pairs = facility_pairs(inst, settings["k_nearest_sources"])
    F = inst["F"]                                    # chain: allowed second destinations from g1
    # "nearest" second yards: lanes are not symmetric, so take the n nearest by distance
    # from g1 AND the n nearest by distance to g1 (union)
    out_ = {g1: sorted((g for g in F if g != g1), key=lambda g: (lane(inst, g1, g)[0], g)) for g1 in F}
    in_ = {g1: sorted((g for g in F if g != g1), key=lambda g: (lane(inst, g, g1)[0], g)) for g1 in F}
    n2 = settings["chain_next"]
    next_dest = {g1: sorted(set(out_[g1][:n2]) | set(in_[g1][:n2])) if n2 is not None else out_[g1]
                 for g1 in F}
    # second move after a drop at g1: to g2 in next_dest[g1], picking up at g1 itself (chain)
    # or at any allowed source of g2 (bobtail from g1 to that source first)
    sources_of = {g: [] for g in F}
    for f, g in pairs:
        sources_of[g].append(f)
    second = {g1: sorted({(g1, g2) for g2 in next_dest[g1]}
                         | {(f2, g2) for g2 in next_dest[g1] for f2 in sources_of[g2]})
              for g1 in F}
    seqs2 = [((f, g), mv2) for f, g in pairs for mv2 in second[g]]
    # simplified: tours after the last demand day cannot help (no excess cost);
    # main: later days can still clear over-full yards
    last_day = inst["demand_days"] if not inst["is_main"] else inst["H"] // 24 - 1

    rows = []
    for k, row in inst["tractors"].iterrows():
        d_open, d_close = opening_hours(inst, row["home_depot_id"])
        dep = row["home_depot_id"]          # quick screen: 2-move tours whose driving alone is too long
        def drive2(s):
            (f1, g1), (f2, g2) = s
            return sum(lane(inst, a, b)[1] for a, b in [(dep, f1), (f1, g1), (g1, f2), (f2, g2), (g2, dep)])
        my_seqs2 = [list(s) for s in seqs2 if drive2(s) <= row["max_drive_hours"] + 1e-9]
        for day in range(last_day + 1):
            depart = 24 * day + max(row["shift_start_hour"], d_open)       # R11
            first = depart
            latest = 24 * day + min(row["shift_end_hour"], d_close)
            while depart < latest:
                chain_ok = settings["chains"] and abs(((depart - first) / settings["chain_step"]) % 1) < 1e-9
                for f, g in pairs:
                    r1 = follow_tour(inst, row, day, depart, [(f, g)])
                    if r1 is not None:
                        rows.append((k, row, day, depart, r1))
                if chain_ok:
                    for legs in my_seqs2:
                        r2 = follow_tour(inst, row, day, depart, legs)
                        if r2 is not None:
                            rows.append((k, row, day, depart, r2))
                depart += settings["step"]

    recs = []
    for k, row, day, depart, (mv, ret, drive, bob, tow) in rows:
        key = tuple(x for m in mv for x in (m[0], m[1], m[2], m[3]))
        recs.append({"tractor": k, "driver": row["driver_name"], "depot": row["home_depot_id"],
                     "day": day, "depart": depart, "return": ret, "drive_hours": drive,
                     "duty_hours": ret - depart, "bobtail_km": bob, "towing_km": tow,
                     "n_moves": len(mv), "_key": key, "_moves": mv,
                     **price(row, depart, ret, bob, tow, len(mv))})
    df = pd.DataFrame(recs)
    # same tractor, day and same hook/drop times = same tour; keep the cheaper (less waiting)
    df = df.sort_values("cost").drop_duplicates(["tractor", "day", "_key"])
    # main: for each depot, day and identical tour, keep the n cheapest tractors
    n = settings["tractors_per_option"]
    if n is not None:
        df = df.sort_values("cost").groupby(["depot", "day", "_key"], sort=False).head(n)
    df = df.sort_values(["day", "tractor", "depart", "_key"]).reset_index(drop=True)
    df.index = [f"R{i:06d}" for i in range(len(df))]
    df.index.name = "tour_id"

    mrows = []
    for r, mv in df["_moves"].items():
        for i, (f, g, hook, drop, drop_end) in enumerate(mv, start=1):
            mrows.append({"move_id": f"{r}-{i}", "tour_id": r, "seq": i, "from": f, "to": g,
                          "hook": hook, "drop_start": drop, "drop_end": drop_end})
    moves = pd.DataFrame(mrows).set_index("move_id")
    tours = df.drop(columns=["_key", "_moves"])
    return tours, moves


def readme_check(inst):
    """README example: T01 leaves D01 at 06:00 on Day 0, moves an empty F05 -> F01. Must cost CAD 1,030."""
    row = inst["tractors"].loc["T01"]
    mv, ret, drive, bob, tow = follow_tour(inst, row, 0, 6.0, [("F05", "F01")])
    c = price(row, 6.0, ret, bob, tow, 1)["cost"]
    assert abs(c - 1030) < 1e-6 and (mv[0][2], mv[0][4], ret) == (8.5, 12.25, 15.0), (mv, ret, c)
    return c




Step 3: The Gurobi model

Binary variables choose which tours run (z) and how each shipment gets its empty (x). Usable, physical and loaded counts (U, P, L), and excess empties (E) on the main file, are tracked every 15 minutes. Constraints: (1) every shipment covered once, (2) a moved empty serves at most one shipment, (3) one shift per tractor per day, (4)–(6) balances for usable, physical and loaded trailers, (7) yard capacity, (8) the empty allowance (charged on the main file).

In [4]:
"""
MODULE 3 - GUROBI MODEL AND PLAN EXPORT (owner: Michael)

Takes the shipment data (Module 1) and the candidate tours (Module 2), lets Gurobi
pick the cheapest legal plan, and writes the plan file the checker reads.

Six-step class pattern: indices -> addVars -> setObjective -> addConstrs (one family
per rule) -> optimize -> check status -> read .X

Run:
  python code/model.py data/project2_simplified.xlsx results/plan_simplified.xlsx
  python code/model.py data/project2_main.xlsx       results/plan_main.xlsx
Options for the penalty analysis: --excess-mult 2  --late-mult 0.5  --time-limit 3600
"""
import argparse, time
from collections import defaultdict
import pandas as pd
import gurobipy as gp
from gurobipy import GRB


LATE_CAP = 12.0        # main file: an empty may be ready at most 12 h after the deadline
EARLY = {False: 48.0, True: 6.0}   # a move is tied to a shipment if it drops at most this many h
                                   # before the deadline (simplified: 48, main: 6). An empty that
                                   # arrives earlier is used as own stock: same plan, same cost.


def q(t):
    """Absolute hours -> quarter-hour step (14.25 h -> 57)."""
    return int(round(4 * t))


# ---------------------------------------------------------------------------
# Ways to cover each shipment (computed before the model, like a coverage table)
# ---------------------------------------------------------------------------
def coverage_options(inst, ships, moves):
    """Each option w = (shipment, kind, ref, ready, late_cost, timeline).
    kind 'own': an empty already at the origin, reserved at ready time s.
    kind 'move': the empty dropped by move m, ready at its drop end.
    Simplified: ready = deadline only. Main: also every quarter hour up to 12 h late (R2, R4)."""
    late = inst["late_allowed"]
    opts = []
    for j, s in ships.iterrows():
        D, rate = s["deadline"], s["late_rate"]
        slots = [D + 0.25 * i for i in range(int(4 * LATE_CAP) + 1)] if late else [D]
        for r in slots:
            t = shipment_times(inst, j, r)
            if t["release"] <= inst["H"]:                               # R5
                opts.append((j, "own", None, r, rate * (r - D), t))
        ok = ((moves["to"] == s["origin"]) & (moves["drop_end"] >= D - EARLY[inst["is_main"]])
              & (moves["drop_end"] <= D + (LATE_CAP if late else 0)))
        for m, e in moves.loc[ok, "drop_end"].items():
            t = shipment_times(inst, j, e)
            if t["release"] <= inst["H"]:
                opts.append((j, "move", m, e, rate * max(0.0, e - D), t))
    return opts


# ---------------------------------------------------------------------------
# Build the model
# ---------------------------------------------------------------------------
def build_model(inst, ships, tours, moves, excess_mult=1.0, late_mult=1.0):
    m = gp.Model("empty_repositioning")
    main = inst["is_main"]

    # ---- Step 1: indices --------------------------------------------------
    F, J, R = inst["F"], list(ships.index), list(tours.index)
    T = range(q(inst["H"]))                    # quarter hours; step t = interval [t/4, t/4 + 0.25)
    opts = coverage_options(inst, ships, moves)
    W = range(len(opts))                       # coverage options
    W_of_j = defaultdict(list)                 # options of each shipment
    W_of_move = defaultdict(list)              # options that use the empty of move m
    for w, (j, kind, ref, *_rest) in enumerate(opts):
        W_of_j[j].append(w)
        if kind == "move":
            W_of_move[ref].append(w)

    # ---- Step 2: variables ------------------------------------------------
    z = m.addVars(R, vtype=GRB.BINARY, name="z")        # tour r is driven
    x = m.addVars(W, vtype=GRB.BINARY, name="x")        # shipment covered by option w
    U = m.addVars(F, T, lb=0, name="U")                 # usable, unpromised empties
    P = m.addVars(F, T, lb=0, name="P")                 # physical empties
    L = m.addVars(F, T, lb=0, name="L")                 # loaded trailers in the yard
    E = m.addVars(F, T, lb=0, name="E") if main else None   # empties above the allowance

    # ---- Step 3: objective ------------------------------------------------
    tour_cost = gp.quicksum(tours.at[r, "cost"] * z[r] for r in R)
    late_cost = gp.quicksum(late_mult * opts[w][4] * x[w] for w in W if opts[w][4] > 0)
    loaded = ships["loaded_cost"].sum()                  # same in every plan
    obj = tour_cost + late_cost + loaded
    if main:
        beta = inst["facilities"]["excess_penalty_cad_per_trailer_hour"] * excess_mult
        obj += gp.quicksum(0.25 * beta[g] * E[g, t] for g in F for t in T)
    m.setObjective(obj, GRB.MINIMIZE)

    # ---- Step 4: constraints ---------------------------------------------
    # (1) every shipment covered exactly once, by own stock or by one moved empty   R1, R2
    m.addConstrs((gp.quicksum(x[w] for w in W_of_j[j]) == 1 for j in J), name="cover")

    # (2) a moved empty serves at most one shipment, and only if its tour runs       R1, R6
    tour_of = moves["tour_id"]
    m.addConstrs((gp.quicksum(x[w] for w in W_of_move[mv]) <= z[tour_of[mv]] for mv in W_of_move),
                 name="link")

    # (3) one tour per tractor per day                                               R10
    groups = tours.groupby(["tractor", "day"]).groups
    m.addConstrs((gp.quicksum(z[r] for r in groups[k, d]) <= 1 for (k, d) in groups), name="one_shift")

    # Events: (yard, quarter hour) -> list of linear terms, plus fixed numbers
    dU, dP, dL = defaultdict(list), defaultdict(list), defaultdict(list)
    cU, cP = defaultdict(float), defaultdict(float)
    NT = len(T)
    def at(t):
        return q(t) if q(t) < NT else None      # events at or after the horizon end are dropped

    for g, n in inst["trailers"].groupby("facility_id").size().items():
        cU[g, 0] += n; cP[g, 0] += n           # initial empties (available_from = 0)
    for mv, rw in moves.iterrows():           # every move of every candidate tour
        zr = z[rw["tour_id"]]
        dU[rw["from"], at(rw["hook"])].append(-zr)               # hook starts: not usable
        dP[rw["from"], at(rw["hook"] + 0.25)].append(-zr)        # leaves the yard
        dP[rw["to"], at(rw["drop_start"])].append(zr)            # in the yard from drop start
        promised = gp.quicksum(x[w] for w in W_of_move.get(mv, []))
        dU[rw["to"], at(rw["drop_end"])].append(zr - promised)   # usable unless promised
    for w, (j, kind, ref, ready, lc, t) in enumerate(opts):
        o, d = ships.at[j, "origin"], ships.at[j, "destination"]
        if kind == "own":
            dU[o, at(ready)].append(-x[w])                       # reserved at its ready time
        dP[o, at(t["load_start"])].append(-x[w])                 # stops being an empty
        dL[o, at(t["load_start"])].append(x[w])                  # being loaded
        dL[o, at(t["loaded_depart"])].append(-x[w])              # carrier leaves
        dL[d, at(t["service_start"])].append(x[w])               # dropped and unloaded
        if at(t["release"]) is not None:
            dL[d, at(t["release"])].append(-x[w])
            dP[d, at(t["release"])].append(x[w])                 # an empty again
            dU[d, at(t["release"])].append(x[w])

    def prev(X, g, t):
        return X[g, t - 1] if t > 0 else 0

    # (4) usable empties: balance, never negative (lower bound 0)                    R6
    m.addConstrs((U[g, t] == prev(U, g, t) + cU[g, t] + gp.quicksum(dU[g, t]) for g in F for t in T),
                 name="usable")
    # (5) physical empties: balance                                                  R7
    m.addConstrs((P[g, t] == prev(P, g, t) + cP[g, t] + gp.quicksum(dP[g, t]) for g in F for t in T),
                 name="physical")
    # (6) loaded trailers in the yard: balance                                      R9
    m.addConstrs((L[g, t] == prev(L, g, t) + gp.quicksum(dL[g, t]) for g in F for t in T), name="loaded")
    # (7) yard capacity, always hard                                                 R9
    cap = inst["facilities"]["trailer_capacity"]
    m.addConstrs((P[g, t] + L[g, t] <= cap[g] for g in F for t in T), name="capacity")
    # (8) allowance: hard on the simplified file, charged on the main file           R8
    allow = inst["facilities"]["empty_allowance"]
    if main:
        m.addConstrs((E[g, t] >= P[g, t] - allow[g] for g in F for t in T), name="excess")
    else:
        m.addConstrs((P[g, t] <= allow[g] for g in F for t in T), name="allowance")

    return m, {"z": z, "x": x, "U": U, "P": P, "L": L, "E": E, "opts": opts}


# ---------------------------------------------------------------------------
# Solve with Gurobi and read the answer
# ---------------------------------------------------------------------------
def solve(m, v, time_limit):
    m.Params.MIPGap = 1e-4
    m.Params.TimeLimit = time_limit
    t0 = time.time()
    m.optimize()
    info = {"solver": "Gurobi " + ".".join(map(str, gp.gurobi.version())),
            "status": {GRB.OPTIMAL: "OPTIMAL", GRB.TIME_LIMIT: "TIME_LIMIT"}.get(m.Status, str(m.Status)),
            "runtime_s": round(time.time() - t0, 1)}
    # always check the status before reading any value
    if not (m.Status == GRB.OPTIMAL or (m.Status == GRB.TIME_LIMIT and m.SolCount > 0)):
        raise RuntimeError(f"No plan to report: Gurobi status {m.Status}")
    info.update(objective=m.ObjVal, gap=m.MIPGap, variables=m.NumVars, binaries=m.NumBinVars,
                constraints=m.NumConstrs)
    vals = {k: {i: var.X for i, var in v[k].items()} for k in ["z", "x", "E"] if v[k] is not None}
    return vals, info


# ---------------------------------------------------------------------------
# Plan: name the physical trailers, then write the sheets
# ---------------------------------------------------------------------------
def extract_plan(inst, ships, tours, moves, v, vals, info):
    opts = v["opts"]
    chosen = [r for r in tours.index if vals["z"][r] > 0.5]
    used = moves[moves["tour_id"].isin(chosen)].copy()
    picked = [opts[w] for w, val in vals["x"].items() if val > 0.5]
    by_ship = {o[0]: o for o in picked}
    promise = {o[2]: o[0] for o in picked if o[1] == "move"}       # move -> shipment

    # Replay in time order; at the same moment trailers arrive before any is taken.
    ev = [(row["available_from"], 0, "arrive", (row["facility_id"], tr)) for tr, row in inst["trailers"].iterrows()]
    ev += [(rw["hook"], 1, "hook", mv) for mv, rw in used.iterrows()]
    ev += [(rw["drop_end"], 0, "drop", mv) for mv, rw in used.iterrows()]
    ev += [(o[3], 1, "reserve", o[0]) for o in picked if o[1] == "own"]
    free = defaultdict(list)                 # yard -> [(since, trailer)]
    tr_move, tr_ship = {}, {}
    while ev:
        ev.sort(key=lambda e: (e[0], e[1]))
        t, _, kind, data = ev.pop(0)
        if kind == "arrive":
            free[data[0]].append((t, data[1]))
        elif kind == "hook":
            f = used.at[data, "from"]; free[f].sort()
            tr_move[data] = free[f].pop(0)[1]
        elif kind == "drop":
            tr = tr_move[data]
            if data in promise:
                j = promise[data]; tr_ship[j] = tr
                ev.append((by_ship[j][5]["release"], 0, "arrive", (ships.at[j, "destination"], tr)))
            else:
                free[used.at[data, "to"]].append((t, tr))
        elif kind == "reserve":
            o = ships.at[data, "origin"]; free[o].sort()
            tr_ship[data] = free[o].pop(0)[1]
            ev.append((by_ship[data][5]["release"], 0, "arrive", (ships.at[data, "destination"], tr_ship[data])))

    used["trailer"] = [tr_move[mv] for mv in used.index]
    used["covers_shipment"] = [promise.get(mv, "") for mv in used.index]
    used["tractor"] = [tours.at[r, "tractor"] for r in used["tour_id"]]
    used = used[["tour_id", "seq", "tractor", "trailer", "from", "to", "hook", "drop_start", "drop_end",
                 "covers_shipment"]].sort_values(["hook", "tractor"])
    tsheet = tours.loc[chosen].copy()

    srows = []
    for j in ships.index:
        jj, kind, ref, ready, lc, t = by_ship[j]
        srows.append({"shipment_id": j, "covered_by": "own stock" if kind == "own" else ref,
                      "trailer": tr_ship[j], "origin": ships.at[j, "origin"],
                      "destination": ships.at[j, "destination"], "deadline": ships.at[j, "deadline"],
                      "ready": ready, "late_hours": t["late_hours"], "late_penalty": lc,
                      "load_start": t["load_start"], "loaded_depart": t["loaded_depart"],
                      "service_start": t["service_start"], "release": t["release"],
                      "loaded_service": ships.at[j, "loaded_cost"]})
    ssheet = pd.DataFrame(srows).set_index("shipment_id")

    beta = inst["facilities"]["excess_penalty_cad_per_trailer_hour"]
    E = vals.get("E", {})
    s = {"bobtail": tsheet["cost_bobtail"].sum(), "empty_towing": tsheet["cost_towing"].sum(),
         "duty_time": tsheet["cost_duty"].sum(), "move_fees": tsheet["cost_fees"].sum(),
         "loaded_service": ssheet["loaded_service"].sum(), "late_penalty": ssheet["late_penalty"].sum(),
         "excess_penalty": sum(0.25 * beta[g] * val for (g, t), val in E.items())}
    s["total_cost"] = sum(s.values())
    s.update({"bobtail_km": tsheet["bobtail_km"].sum(), "towing_km": tsheet["towing_km"].sum(),
              "shipments_on_time": int((ssheet["late_hours"] <= 1e-9).sum()),
              "late_hours": ssheet["late_hours"].sum(),
              "excess_trailer_hours": sum(0.25 * val for val in E.values()),
              "covered_by_own_stock": int((ssheet["covered_by"] == "own stock").sum()),
              "empty_moves": len(used), "tractor_shifts": len(tsheet),
              "two_move_tours": int((tsheet["n_moves"] == 2).sum()),
              "candidate_tours": len(tours), **info})
    return tsheet, used, ssheet, pd.Series(s, name="value")


def write_plan(path, tsheet, msheet, ssheet, summary):
    with pd.ExcelWriter(path) as xw:
        tsheet.to_excel(xw, sheet_name="tours")
        msheet.to_excel(xw, sheet_name="moves")
        ssheet.to_excel(xw, sheet_name="shipments")
        summary.to_frame().to_excel(xw, sheet_name="summary")


def prepare(data_path):
    inst = load_instance(data_path)
    ships = shipment_table(inst)
    tours, moves = build_tours(inst, SETTINGS["main" if inst["is_main"] else "simplified"])
    return inst, ships, tours, moves





Step 4: Solve the main file and show the results

Solves with MIPGap 0.01% and a 3600 s time limit, writes the plan to results/plan_main.xlsx, and shows the cost breakdown, Gurobi status, gap and runtime.

In [5]:
   inst, ships, tours, moves = prepare("data/project2_main.xlsx")
   m, v = build_model(inst, ships, tours, moves)
   vals, info = solve(m, v, 3600)
   tsheet, msheet, ssheet, summary = extract_plan(inst, ships, tours, moves, v, vals, info)
   write_plan("results/plan_main.xlsx", tsheet, msheet, ssheet, summary)
   summary

Set parameter Username
Set parameter LicenseID to value 2884335
Academic license - for non-commercial use only - expires 2027-11-12
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 27.0.0 26A434)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 182669 rows, 377133 columns and 2571507 nonzeros (Min)
Model fingerprint: 0x02d47858
Model has 238047 linear objective coefficients and an objective constant of 86815
Variable types: 92160 continuous, 284973 integer (284973 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve removed 110001 rows and 142824 columns
Presolve time: 2.20s
Presolved: 72668 rows, 234309 columns, 1522945 nonzeros
Variable types: 0 continuo

bobtail                      29020.27
empty_towing                 24317.51
duty_time                     19945.0
move_fees                        1567
loaded_service                86815.0
late_penalty                    515.0
excess_penalty                 3525.0
total_cost                  165704.78
bobtail_km                    27748.0
towing_km                     16144.0
shipments_on_time                 164
late_hours                        6.0
excess_trailer_hours           176.25
covered_by_own_stock              164
empty_moves                       100
tractor_shifts                     78
two_move_tours                     22
candidate_tours                138930
solver                  Gurobi 13.0.3
status                        OPTIMAL
runtime_s                         4.4
objective                   165704.78
gap                               0.0
variables                      377133
binaries                       284973
constraints                    182669
Name: value,

Schedule of empty moves

The first 20 empty moves of the plan: tour, tractor, trailer, from, to, hook start, drop start and drop end (absolute hours), and which shipment each move covers. The full schedule is in results/plan_main.xlsx

In [6]:
   msheet.head(20)

,tour_id,seq,tractor,trailer,from,to,hook,drop_start,drop_end,covers_shipment
move_id,,,,,,,,,,
R009108-1,R009108,1,T16,TR_F21_01,F21,F20,8.25,10.75,11.00,
R001203-1,R001203,1,T04,TR_F19_01,F19,F09,8.50,11.00,11.25,
R003195-1,R003195,1,T06,TR_F06_01,F06,F08,8.50,11.25,11.50,
R004608-1,R004608,1,T07,TR_F17_01,F17,F04,8.50,11.00,11.25,
R005444-1,R005444,1,T09,TR_F21_02,F21,F10,8.50,11.00,11.25,
R011378-1,R011378,1,T21,TR_F22_01,F22,F05,8.50,11.25,11.50,
R011568-1,R011568,1,T24,TR_F19_02,F19,F09,8.50,11.00,11.25,
R012159-1,R012159,1,T30,TR_F06_02,F06,F08,8.50,11.25,11.50,
R002511-1,R002511,1,T05,TR_F23_01,F23,F04,8.75,11.25,11.50,


Step 5: Penalty analysis

Re-solves the main file with the penalty rates scaled in the objective: excess-empty rate ×0 and ×2, late rate ×0.5 and ×2. It reuses the tours built above. Costs shown are recomputed at the workbook rates, so the plans can be compared directly.

In [7]:
   runs = {"excess0": (0, 1), "excess2": (2, 1), "late0.5": (1, 0.5), "late2": (1, 2)}
   penalty = {}
   for name, (em, lm) in runs.items():
       m2, v2 = build_model(inst, ships, tours, moves, em, lm)
       vals2, info2 = solve(m2, v2, 3600)
       t2, mv2, s2, sum2 = extract_plan(inst, ships, tours, moves, v2, vals2, info2)
       write_plan(f"results/plan_main_{name}.xlsx", t2, mv2, s2, sum2)
       penalty[name] = sum2
   pd.DataFrame(penalty)

Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 27.0.0 26A434)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 182669 rows, 377133 columns and 2571507 nonzeros (Min)
Model fingerprint: 0x45241b76
Model has 215007 linear objective coefficients and an objective constant of 86815
Variable types: 92160 continuous, 284973 integer (284973 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+01, 2e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve removed 120332 rows and 153091 columns
Presolve time: 2.48s
Presolved: 62337 rows, 224042 columns, 1507239 nonzeros
Variable types: 0 continuous, 224042 integer (210127 binary)
Deterministic concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier 

,excess0,excess2,late0.5,late2
bobtail,21212.16,29839.9,28794.9,29453.89
empty_towing,14707.26,24282.68,22879.91,24506.28
duty_time,13397.0,20310.0,19263.5,20164.5
move_fees,988,1548,1481,1579
loaded_service,86815.0,86815.0,86815.0,86815.0
late_penalty,2945.0,415.0,2800.0,30.0
excess_penalty,116825.0,2745.0,4425.0,3520.0
total_cost,256889.42,165955.58,166459.31,166068.67
bobtail_km,20637.0,28534.0,27500.0,28129.0
towing_km,9622.0,16110.0,15168.0,16258.0


Step 6: Nearest-empty baseline

A simple documented rule to compare against, with every tie-break stated so anyone can rerun it. No solver is used.

Take shipments in deadline order (ties by shipment_id).
If a free empty is at the origin by the deadline, use it, with no move. If there are several, use the one free the longest, then the lowest trailer_id.
Otherwise, try source facilities from nearest to farthest by lane distance to the origin (ties by facility_id). At each source, try its free empties in the order they became free (ties by trailer_id). For each empty, try days from the deadline day back to Day 0 (latest first). On each day, try the tractors in tractor_id order that have no tour yet that day, leaving at shift start (or later if the empty isn’t free yet). Take the first combination that obeys every tractor rule and yard limit and drops by the deadline. One move per tractor shift.
Main file only: if nothing drops by the deadline, take the combination with the smallest lateness (up to 12 h, same tie-breaking order) and charge the late penalty.

Excess empties are never managed by the rule; on the main file they are just charged.

In [8]:
"""
NEAREST-EMPTY BASELINE

The rule, with every tie-break, so anyone can rerun it and get the same plan:
 1. Take shipments in deadline order (absolute hours); ties by shipment_id.
 2. If a free empty is at the origin by the deadline, use it: no move. If several,
    the one free the longest, then the lowest trailer_id.
 3. Otherwise try source facilities from nearest to farthest by lane distance to the
    origin (ties: facility_id). At each source, try its free empties in order of
    when they became free (ties: trailer_id). For each, try days from the deadline
    day back to Day 0 (latest first), and on each day the tractors in tractor_id
    order that have no tour yet that day. The tractor leaves at shift start, or later
    if the empty is not free yet. Take the first combination that obeys every tractor
    rule and yard limit and drops by the deadline. One move per tractor shift.
 4. Main file only: if nothing drops by the deadline, take the combination with the
    smallest lateness (up to 12 h, same tie-breaking order) and charge the late penalty.
Excess empties are never managed by the rule; on the main file they are just charged.

Run: python code/baseline.py data/project2_main.xlsx results/baseline_main.xlsx
"""
import sys, math
from collections import defaultdict
import numpy as np
import pandas as pd

LATE_CAP = 12.0


def q(t):
    return int(round(4 * t))


def run_baseline(inst):
    ships = shipment_table(inst).sort_index().sort_values("deadline", kind="stable")   # deadline, then id
    F, NT = inst["F"], q(inst["H"])
    fac = inst["facilities"]
    P = {g: np.zeros(NT) for g in F}          # physical empties per quarter hour
    Lc = {g: np.zeros(NT) for g in F}         # loaded trailers per quarter hour
    where = {}                                # trailer -> (site, free since)
    for tr, row in inst["trailers"].iterrows():
        where[tr] = (row["facility_id"], float(row["available_from"]))
        P[row["facility_id"]][q(row["available_from"]):] += 1
    busy = set()                              # (tractor, day) already used
    tours, moves, srows = [], [], []

    def change(arr, g, t0, t1, d):
        arr[g][q(t0):min(q(t1), NT)] += d

    def fits_yards(changes):
        """Apply tentative changes, check capacity (both files) and allowance (simplified), undo if bad."""
        for arr, g, t0, t1, d in changes:
            change(arr, g, t0, t1, d)
        ok = True
        for _, g, t0, t1, d in changes:
            if d > 0:
                a, b = q(t0), min(q(t1), NT)
                if (P[g][a:b] + Lc[g][a:b] > fac.at[g, "trailer_capacity"] + 1e-9).any():
                    ok = False
                if not inst["is_main"] and (P[g][a:b] > fac.at[g, "empty_allowance"] + 1e-9).any():
                    ok = False
        if not ok:
            for arr, g, t0, t1, d in changes:
                change(arr, g, t0, t1, -d)
        return ok

    def shipment_changes(j, t):
        o, d = ships.at[j, "origin"], ships.at[j, "destination"]
        return [(P, o, t["load_start"], inst["H"], -1),            # stops being an empty
                (Lc, o, t["load_start"], t["loaded_depart"], 1),
                (Lc, d, t["service_start"], t["release"], 1),
                (P, d, t["release"], inst["H"], 1)]                 # an empty again

    for j, s in ships.iterrows():
        D, o = s["deadline"], s["origin"]
        # step 2: own stock at the origin
        local = sorted((since, tr) for tr, (site, since) in where.items() if site == o and since <= D)
        done = False
        for since, tr in local:
            t = shipment_times(inst, j, D)
            if fits_yards(shipment_changes(j, t)):
                where[tr] = (s["destination"], t["release"])
                srows.append({"shipment_id": j, "covered_by": "own stock", "trailer": tr, "ready": D, **t})
                done = True
                break
        if done:
            continue
        # step 3 (and 4 on the main file): nearest source, then trailer, then day, then tractor
        sources = sorted((f for f in F if f != o), key=lambda f: (lane(inst, f, o)[0], f))
        best = None                                           # (lateness, order, option)
        order = 0
        for f in sources:
            cands = sorted((since, tr) for tr, (site, since) in where.items() if site == f)
            for since, tr in cands:
                days = range(int(D // 24), -1, -1)
                if inst["late_allowed"]:
                    days = list(days) + [int(D // 24) + 1]
                for day in days:
                    for k, row in inst["tractors"].iterrows():
                        if (k, day) in busy:
                            continue
                        d_open, _ = opening_hours(inst, row["home_depot_id"])
                        start = 24 * day + max(row["shift_start_hour"], d_open)
                        dep = max(start, math.ceil(4 * (since - lane(inst, row["home_depot_id"], f)[1])) / 4)
                        res = follow_tour(inst, row, day, dep, [(f, o)])
                        if res is None:
                            continue
                        mv, ret, drive, bob, tow = res
                        hook, dstart, dend = mv[0][2], mv[0][3], mv[0][4]
                        late = max(0.0, dend - D)
                        if late > 0 and (not inst["late_allowed"] or late > LATE_CAP):
                            continue
                        order += 1
                        if best is not None and (late, order) >= (best[0], best[1]):
                            continue
                        t = shipment_times(inst, j, dend)
                        if t["release"] > inst["H"]:
                            continue
                        ch = [(P, f, hook + 0.25, inst["H"], -1), (P, o, dstart, inst["H"], 1)] + shipment_changes(j, t)
                        if not fits_yards(ch):
                            continue
                        for arr, g, t0, t1, dd in ch:     # undo; re-applied if this one wins
                            change(arr, g, t0, t1, -dd)
                        best = (late, order, (f, tr, day, k, row, dep, res, t, ch))
                        if late == 0:
                            break
                    if best is not None and best[0] == 0:
                        break
                if best is not None and best[0] == 0:
                    break
            if best is not None and best[0] == 0:
                break
        if best is None:
            raise RuntimeError(f"baseline found no way to cover {j}")
        f, tr, day, k, row, dep, (mv, ret, drive, bob, tow), t, ch = best[2]
        for arr, g, t0, t1, dd in ch:
            change(arr, g, t0, t1, dd)
        busy.add((k, day))
        rid = f"B{len(tours) + 1:04d}"
        tours.append({"tour_id": rid, "tractor": k, "driver": row["driver_name"], "depot": row["home_depot_id"],
                      "day": day, "depart": dep, "return": ret, "drive_hours": drive, "duty_hours": ret - dep,
                      "bobtail_km": bob, "towing_km": tow, "n_moves": 1, **price(row, dep, ret, bob, tow, 1)})
        moves.append({"move_id": f"{rid}-1", "tour_id": rid, "seq": 1, "tractor": k, "trailer": tr,
                      "from": f, "to": o, "hook": mv[0][2], "drop_start": mv[0][3], "drop_end": mv[0][4],
                      "covers_shipment": j})
        where[tr] = (s["destination"], t["release"])
        srows.append({"shipment_id": j, "covered_by": f"{rid}-1", "trailer": tr, **t})

    tsheet = pd.DataFrame(tours).set_index("tour_id")
    msheet = pd.DataFrame(moves).set_index("move_id")
    sh = pd.DataFrame(srows).set_index("shipment_id").loc[sorted(r["shipment_id"] for r in srows)]
    base = shipment_table(inst)
    for c in ["origin", "destination", "deadline"]:
        sh[c] = base.loc[sh.index, c]
    sh["late_penalty"] = sh["late_hours"] * base.loc[sh.index, "late_rate"]
    sh["loaded_service"] = base.loc[sh.index, "loaded_cost"]
    sh = sh[["covered_by", "trailer", "origin", "destination", "deadline", "ready", "late_hours", "late_penalty",
             "load_start", "loaded_depart", "service_start", "release", "loaded_service"]]
    allow = fac["empty_allowance"]; beta = fac["excess_penalty_cad_per_trailer_hour"]
    exc_h = sum(0.25 * np.clip(P[g] - allow[g], 0, None).sum() for g in F)
    exc_c = sum(0.25 * beta[g] * np.clip(P[g] - allow[g], 0, None).sum() for g in F) if inst["is_main"] else 0.0
    summ = {"bobtail": tsheet["cost_bobtail"].sum(), "empty_towing": tsheet["cost_towing"].sum(),
            "duty_time": tsheet["cost_duty"].sum(), "move_fees": tsheet["cost_fees"].sum(),
            "loaded_service": sh["loaded_service"].sum(), "late_penalty": sh["late_penalty"].sum(),
            "excess_penalty": exc_c}
    summ["total_cost"] = sum(summ.values())
    summ.update({"bobtail_km": tsheet["bobtail_km"].sum(), "towing_km": tsheet["towing_km"].sum(),
                 "shipments_on_time": int((sh["late_hours"] <= 1e-9).sum()), "late_hours": sh["late_hours"].sum(),
                 "excess_trailer_hours": exc_h if inst["is_main"] else 0.0,
                 "covered_by_own_stock": int((sh["covered_by"] == "own stock").sum()),
                 "empty_moves": len(msheet), "tractor_shifts": len(tsheet), "two_move_tours": 0,
                 "solver": "nearest-empty rule (no solver)"})
    return tsheet, msheet, sh, pd.Series(summ, name="value")



Runs the baseline on the main file, writes the plan to results/baseline_main.xlsx, and shows its summary.

In [9]:
inst_b = load_instance("data/project2_main.xlsx")
out = run_baseline(inst_b)
write_plan("results/baseline_main.xlsx", *out)
out[3]

bobtail                                       32268.03
empty_towing                                  18993.59
duty_time                                      20131.0
move_fees                                         1186
loaded_service                                 86815.0
late_penalty                                       0.0
excess_penalty                                118385.0
total_cost                                   277778.62
bobtail_km                                     30693.0
towing_km                                      12074.0
shipments_on_time                                  168
late_hours                                         0.0
excess_trailer_hours                           5919.25
covered_by_own_stock                                92
empty_moves                                         76
tractor_shifts                                      76
two_move_tours                                       0
solver                  nearest-empty rule (no solver)
Name: valu

Baseline vs Gurobi

Side-by-side cost, kilometres and service on the main file. The Gurobi plan avoids most of the baseline’s excess-empty penalty and covers more shipments with empties already at the origin.

In [10]:
rows = ["bobtail", "empty_towing", "duty_time", "move_fees", "loaded_service", "late_penalty",
        "excess_penalty", "total_cost", "bobtail_km", "towing_km", "shipments_on_time",
        "late_hours", "excess_trailer_hours", "covered_by_own_stock", "empty_moves", "tractor_shifts"]
pd.DataFrame({"Baseline": out[3], "Gurobi": summary}).loc[rows]

,Baseline,Gurobi
bobtail,32268.03,29020.27
empty_towing,18993.59,24317.51
duty_time,20131.0,19945.0
move_fees,1186,1567
loaded_service,86815.0,86815.0
late_penalty,0.0,515.0
excess_penalty,118385.0,3525.0
total_cost,277778.62,165704.78
bobtail_km,30693.0,27748.0
towing_km,12074.0,16144.0
